In [1]:
import ee

# Initialize the Earth Engine library
ee.Authenticate()
ee.Initialize()

# Define your Area of Interest (example: a point geometry)
aoi = ee.Geometry.Point([-122.0841, 37.4219])  # Replace with your coordinates

# Define date range
start_date = '2023-01-01'
end_date = '2023-12-31'

# Load Sentinel-2 Surface Reflectance and Cloud Probability collections
s2_sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi).filterDate(start_date, end_date)
s2_cloud_prob = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filterBounds(aoi).filterDate(start_date, end_date)

# Join the collections on their 'system:index' property
save_best_join = ee.Join.saveFirst(matchKey='cloud_probability')
joined_collection = save_best_join.apply(s2_sr, s2_cloud_prob, ee.Filter.equals(leftField='system:index', rightField='system:index'))

# Function to mask clouds using the joined cloud probability
def mask_clouds(img):
    # Get the corresponding cloud probability image
    cld_prb = ee.Image(img.get('cloud_probability')).select('probability')
    # Create a mask (e.g., keep pixels with less than 40% cloud probability)
    is_not_cloud = cld_prb.lt(40)
    # Apply the mask to the original image
    return img.updateMask(is_not_cloud)

# Function to calculate NDVI
def calc_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi

# Apply cloud masking and calculate NDVI
masked_s2 = joined_collection.map(mask_clouds)
ndvi_collection = masked_s2.map(calc_ndvi)

# Create a median composite
ndvi_median = ndvi_collection.median().clip(aoi)

# Export the final NDVI image as a GeoTIFF to Google Drive
task = ee.batch.Export.image.toDrive(
    image=ndvi_median,
    description='S2_NDVI_Masked_Export',
    folder='GEE_Exports',  # Your Google Drive folder
    fileNamePrefix='S2_NDVI_median',
    region=aoi,
    scale=10,  # 10m resolution
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

# Start the export task (you must run this from the GEE Tasks tab)
task.start()

/home/martin/repositories/RemoteSensing/.venv/lib/python3.14/site-packages/ee/main.py:150: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  lines = filter(lambda x: re.match("^\d+ bytes", x), data.splitlines())


ModuleNotFoundError: No module named 'StringIO'